Student Information Management System (SIMS) - Planning Phase
Functional Requirements
* Add, view, delete students
* Add courses
* Enroll students in courses with grades
* View enrollments
* Export student data (mid-level)
Business Rules
* Unique student emails
* No duplicate enrollments
* Grades must be A–F
* Age > 0
Sample Data
(Write sample students/courses/enrollments here)
Reports Planned
* Enrollments with grades
* Course-wise student count
* Grade distribution

Entity-Relationship Diagram (Draft)
Entities:
* STUDENTS(id, name, age, gender, email)
* COURSES(course_id, course_name)
* ENROLLMENTS(enrollment_id, student_id → STUDENTS.id, course_id → COURSES.course_id, grade)
Relationships:
* One student → Many enrollments
* One course → Many enrollments
* ENROLLMENTS is a bridge (many-to-many) between STUDENTS and COURSES



Phase 2: System Design
Database Tables
1. Students
* id (PK)
* name
* age
* gender
* email
2. Courses
* course_id (PK)
* course_name
3. Enrollments
* enrollment_id (PK)
* student_id (FK to Students)
* course_id (FK to Courses)
* grade
Relationships
* One student can enroll in many courses
* One course can have many students
* Enrollments table bridges students and courses
ER Diagram
(Insert ERD image here or describe as above)







In [ ]:
import sqlite3

# Connect to SQLite database (creates it if not exists)
conn = sqlite3.connect("sims.db")
cursor = conn.cursor()

# Create Users table (admin, student, teacher)
cursor.execute("""
CREATE TABLE IF NOT EXISTS Users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    role TEXT CHECK(role IN ('admin', 'student', 'teacher')) NOT NULL
)
""")

# Create Departments table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Departments (
    department_id INTEGER PRIMARY KEY AUTOINCREMENT,
    department_name TEXT NOT NULL
)
""")

# Create Courses table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
)
""")

# Create Students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    dob TEXT,
    email TEXT,
    department_id INTEGER,
    user_id INTEGER UNIQUE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id),
    FOREIGN KEY(user_id) REFERENCES Users(user_id)
)
""")

# Create Teachers table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Teachers (
    teacher_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT,
    department_id INTEGER,
    user_id INTEGER UNIQUE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id),
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
)
""")

# Create Enrollments table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    course_id INTEGER,
    enrollment_date TEXT,
    FOREIGN KEY (student_id) REFERENCES Students(student_id),
    FOREIGN KEY (course_id) REFERENCES Courses(course_id)
)
""")

conn.commit()
print(" All tables created successfully.")


 All tables created successfully.


In [ ]:
import sqlite3
conn = sqlite3.connect('sims.db')
cursor = conn.cursor()

cursor.executescript("""
-- DROP Tables if they exist
DROP TABLE IF EXISTS Enrollments;
DROP TABLE IF EXISTS Students;
DROP TABLE IF EXISTS Teachers;
DROP TABLE IF EXISTS Courses;
DROP TABLE IF EXISTS Departments;
DROP TABLE IF EXISTS Users;

-- CREATE Tables

CREATE TABLE Departments (
    department_id INTEGER PRIMARY KEY AUTOINCREMENT,
    department_name TEXT UNIQUE NOT NULL
);

CREATE TABLE Courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
);

CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    dob TEXT,
    email TEXT,
    department_id INTEGER,
    user_id INTEGER UNIQUE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id),
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
);

CREATE TABLE Teachers (
    teacher_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT,
    department_id INTEGER,
    user_id INTEGER UNIQUE,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id),
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
);

CREATE TABLE Enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER,
    course_id INTEGER,
    enrollment_date TEXT,
    grade TEXT,
    FOREIGN KEY (student_id) REFERENCES Students(student_id),
    FOREIGN KEY (course_id) REFERENCES Courses(course_id)
);

CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    password TEXT NOT NULL,
    role TEXT CHECK(role IN ('admin', 'student', 'teacher')) NOT NULL
);

-- INSERT Sample Departments
INSERT OR IGNORE INTO Departments (department_name) VALUES ('Computer Science');
INSERT OR IGNORE INTO Departments (department_name) VALUES ('Mathematics');
INSERT OR IGNORE INTO Departments (department_name) VALUES ('Physics');
INSERT OR IGNORE INTO Departments (department_name) VALUES ('Business');

-- INSERT Sample Courses
INSERT OR IGNORE INTO Courses (course_name, department_id) VALUES ('Intro to CS', 1);
INSERT OR IGNORE INTO Courses (course_name, department_id) VALUES ('Data Structures', 1);
INSERT OR IGNORE INTO Courses (course_name, department_id) VALUES ('Calculus I', 2);
INSERT OR IGNORE INTO Courses (course_name, department_id) VALUES ('Marketing Basics', 4);

-- INSERT Sample Students
INSERT OR IGNORE INTO Students (first_name, last_name, dob, email, department_id)
VALUES ('Alice', 'Smith', '2000-01-01', 'alice@example.com', 1);

INSERT OR IGNORE INTO Students (first_name, last_name, dob, email, department_id)
VALUES ('Bob', 'Johnson', '1999-05-12', 'bob@example.com', 2);

-- INSERT Sample Teachers
INSERT OR IGNORE INTO Teachers (first_name, last_name, email, department_id)
VALUES ('John', 'Doe', 'johndoe@university.edu', 1);

-- INSERT Sample Enrollments
INSERT OR IGNORE INTO Enrollments (student_id, course_id, enrollment_date, grade)
VALUES (1, 1, '2024-09-01', 'A');

-- INSERT Sample Users
INSERT OR IGNORE INTO Users (username, password, role)
VALUES ('admin', 'admin123', 'admin');

INSERT OR IGNORE INTO Users (username, password, role)
VALUES ('alice_smith', 'alicepass', 'student');

INSERT OR IGNORE INTO Users (username, password, role)
VALUES ('student1', 'studpass', 'student');

INSERT OR IGNORE INTO Students (first_name, last_name, dob, email, department_id, user_id)
VALUES (
    'Charlie', 'Brown', '2001-03-15', 'charlie@example.com', 1,
    (SELECT user_id FROM Users WHERE username = 'student1')
);

INSERT OR IGNORE INTO Users (username, password, role)
VALUES ('johndoe', 'doepass', 'teacher');
""")

# Now do the linking updates in Python after the script runs

cursor.execute("""
    UPDATE Students SET user_id = (
        SELECT user_id FROM Users WHERE username = 'alice_smith'
    )
    WHERE email = 'alice@example.com'
""")

cursor.execute("""
    UPDATE Teachers SET user_id = (
        SELECT user_id FROM Users WHERE username = 'johndoe'
    )
    WHERE email = 'johndoe@university.edu'
""")

conn.commit()
conn.close()
print(" SIMS database reset and sample data inserted successfully!")


 SIMS database reset and sample data inserted successfully!


In [ ]:
import sqlite3

# CREATE
def create_student(first_name, last_name, dob, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Students (first_name, last_name, dob, email, department_id)
            VALUES (?, ?, ?, ?, ?)
        """, (first_name, last_name, dob, email, department_id))
        conn.commit()
        print(" Student created successfully.")

# READ
def read_students():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT student_id, first_name, last_name, dob, email, department_name
            FROM Students
            JOIN Departments ON Students.department_id = Departments.department_id
        """)
        rows = cursor.fetchall()
        print(" All Students:")
        for row in rows:
            print(row)

# UPDATE
def update_student(student_id, first_name, last_name, dob, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Students
            SET first_name = ?, last_name = ?, dob = ?, email = ?, department_id = ?
            WHERE student_id = ?
        """, (first_name, last_name, dob, email, department_id, student_id))
        conn.commit()
        print(f" Student ID {student_id} updated successfully.")

# DELETE
def delete_student(student_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Students WHERE student_id = ?", (student_id,))
        conn.commit()
        print(f" Student ID {student_id} deleted successfully.")


In [ ]:
# Create new student
create_student("Ethan", "Wong", "2002-05-10", "ethan.wong@example.com", 1)

# List all students
read_students()

# Update an existing student (e.g., ID 1)
update_student(1, "Alice", "Smith", "2000-01-01", "alice.updated@example.com", 1)

# Delete a student by ID (only if ID exists)
delete_student(3)  # Only works if student with ID 3 exists

# Final check
read_students()


 Student created successfully.
 All Students:
(1, 'Alice', 'Smith', '2000-01-01', 'alice@example.com', 'Computer Science')
(2, 'Bob', 'Johnson', '1999-05-12', 'bob@example.com', 'Mathematics')
(3, 'Charlie', 'Brown', '2001-03-15', 'charlie@example.com', 'Computer Science')
(4, 'Ethan', 'Wong', '2002-05-10', 'ethan.wong@example.com', 'Computer Science')
 Student ID 1 updated successfully.
 Student ID 3 deleted successfully.
 All Students:
(1, 'Alice', 'Smith', '2000-01-01', 'alice.updated@example.com', 'Computer Science')
(2, 'Bob', 'Johnson', '1999-05-12', 'bob@example.com', 'Mathematics')
(4, 'Ethan', 'Wong', '2002-05-10', 'ethan.wong@example.com', 'Computer Science')


In [ ]:
import sqlite3

# CREATE Teacher
def create_teacher(first_name, last_name, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Teachers (first_name, last_name, email, department_id)
            VALUES (?, ?, ?, ?)
        """, (first_name, last_name, email, department_id))
        conn.commit()
        print(" Teacher created successfully.")

# READ Teachers
def read_teachers():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT teacher_id, first_name, last_name, email, department_name
            FROM Teachers
            JOIN Departments ON Teachers.department_id = Departments.department_id
        """)
        rows = cursor.fetchall()
        print(" All Teachers:")
        for row in rows:
            print(row)

# UPDATE Teacher
def update_teacher(teacher_id, first_name, last_name, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Teachers
            SET first_name = ?, last_name = ?, email = ?, department_id = ?
            WHERE teacher_id = ?
        """, (first_name, last_name, email, department_id, teacher_id))
        conn.commit()
        print(f" Teacher ID {teacher_id} updated successfully.")

# DELETE Teacher
def delete_teacher(teacher_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Teachers WHERE teacher_id = ?", (teacher_id,))
        conn.commit()
        print(f" Teacher ID {teacher_id} deleted successfully.")


In [ ]:
# Create a new teacher
create_teacher("Sarah", "Connor", "sarah.connor@example.com", 1)

# Read all teachers
read_teachers()

# Update teacher ID 1
update_teacher(1, "John", "Doe", "john.doe@updatedemail.com", 1)

# Delete teacher ID 2 (if exists)
delete_teacher(2)

# Final teachers list
read_teachers()


 Teacher created successfully.
 All Teachers:
(1, 'John', 'Doe', 'johndoe@university.edu', 'Computer Science')
(2, 'Sarah', 'Connor', 'sarah.connor@example.com', 'Computer Science')
 Teacher ID 1 updated successfully.
 Teacher ID 2 deleted successfully.
 All Teachers:
(1, 'John', 'Doe', 'john.doe@updatedemail.com', 'Computer Science')


In [ ]:
import sqlite3

# CREATE Course
def create_course(course_name, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Courses (course_name, department_id)
            VALUES (?, ?)
        """, (course_name, department_id))
        conn.commit()
        print(" Course created successfully.")

# READ Courses
def read_courses():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT course_id, course_name, department_name
            FROM Courses
            JOIN Departments ON Courses.department_id = Departments.department_id
        """)
        rows = cursor.fetchall()
        print(" All Courses:")
        for row in rows:
            print(row)

# UPDATE Course
def update_course(course_id, course_name, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Courses
            SET course_name = ?, department_id = ?
            WHERE course_id = ?
        """, (course_name, department_id, course_id))
        conn.commit()
        print(f" Course ID {course_id} updated successfully.")

# DELETE Course
def delete_course(course_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Courses WHERE course_id = ?", (course_id,))
        conn.commit()
        print(f" Course ID {course_id} deleted successfully.")


In [ ]:
# Create a new course
create_course("Algorithms", 1)

# Read all courses
read_courses()

# Update course ID 1
update_course(1, "Introduction to Computer Science", 1)

# Delete course ID 3 (if exists)
delete_course(3)

# Final courses list
read_courses()


 Course created successfully.
 All Courses:
(1, 'Intro to CS', 'Computer Science')
(2, 'Data Structures', 'Computer Science')
(3, 'Calculus I', 'Mathematics')
(4, 'Marketing Basics', 'Business')
(5, 'Algorithms', 'Computer Science')
 Course ID 1 updated successfully.
 Course ID 3 deleted successfully.
 All Courses:
(1, 'Introduction to Computer Science', 'Computer Science')
(2, 'Data Structures', 'Computer Science')
(4, 'Marketing Basics', 'Business')
(5, 'Algorithms', 'Computer Science')


In [ ]:
import sqlite3

# CREATE Enrollment
def create_enrollment(student_id, course_id, enrollment_date, grade=None):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Enrollments (student_id, course_id, enrollment_date, grade)
            VALUES (?, ?, ?, ?)
        """, (student_id, course_id, enrollment_date, grade))
        conn.commit()
        print(" Enrollment created successfully.")

# READ Enrollments
def read_enrollments():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT enrollment_id, Students.first_name || ' ' || Students.last_name AS student_name,
                   Courses.course_name, enrollment_date, grade
            FROM Enrollments
            JOIN Students ON Enrollments.student_id = Students.student_id
            JOIN Courses ON Enrollments.course_id = Courses.course_id
        """)
        rows = cursor.fetchall()
        print(" All Enrollments:")
        for row in rows:
            print(row)

# UPDATE Enrollment
def update_enrollment(enrollment_id, student_id, course_id, enrollment_date, grade):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Enrollments
            SET student_id = ?, course_id = ?, enrollment_date = ?, grade = ?
            WHERE enrollment_id = ?
        """, (student_id, course_id, enrollment_date, grade, enrollment_id))
        conn.commit()
        print(f" Enrollment ID {enrollment_id} updated successfully.")

# DELETE Enrollment
def delete_enrollment(enrollment_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Enrollments WHERE enrollment_id = ?", (enrollment_id,))
        conn.commit()
        print(f" Enrollment ID {enrollment_id} deleted successfully.")


In [ ]:
# Create a new enrollment
create_enrollment(1, 1, "2025-06-01", "A")

# Read all enrollments
read_enrollments()

# Update enrollment ID 1 (change grade or course)
update_enrollment(1, 1, 2, "2025-06-01", "B+")

# Delete enrollment ID 2 (if exists)
delete_enrollment(2)

# Final enrollments list
read_enrollments()


 Enrollment created successfully.
 All Enrollments:
(1, 'Alice Smith', 'Introduction to Computer Science', '2024-09-01', 'A')
(2, 'Alice Smith', 'Introduction to Computer Science', '2025-06-01', 'A')
 Enrollment ID 1 updated successfully.
 Enrollment ID 2 deleted successfully.
 All Enrollments:
(1, 'Alice Smith', 'Data Structures', '2025-06-01', 'B+')


In [ ]:
import sqlite3

# CREATE User
def create_user(username, password, role):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Users (username, password, role)
            VALUES (?, ?, ?)
        """, (username, password, role))
        conn.commit()
        print(" User created successfully.")

# READ Users
def read_users():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT user_id, username, role FROM Users
        """)
        rows = cursor.fetchall()
        print(" All Users:")
        for row in rows:
            print(row)

# UPDATE User
def update_user(user_id, username, password, role):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Users
            SET username = ?, password = ?, role = ?
            WHERE user_id = ?
        """, (username, password, role, user_id))
        conn.commit()
        print(f" User ID {user_id} updated successfully.")

# DELETE User
def delete_user(user_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Users WHERE user_id = ?", (user_id,))
        conn.commit()
        print(f" User ID {user_id} deleted successfully.")


In [ ]:
# Create a new user
create_user("newuser", "securepassword", "student")

# Read all users
read_users()

# Update user ID 1
update_user(1, "admin", "newadminpass", "admin")

# Delete user ID 3 (if exists)
delete_user(3)

# Final users list
read_users()


 User created successfully.
 All Users:
(1, 'admin', 'admin')
(2, 'alice_smith', 'student')
(3, 'student1', 'student')
(4, 'johndoe', 'teacher')
(5, 'newuser', 'student')
 User ID 1 updated successfully.
 User ID 3 deleted successfully.
 All Users:
(1, 'admin', 'admin')
(2, 'alice_smith', 'student')
(4, 'johndoe', 'teacher')
(5, 'newuser', 'student')


In [ ]:
pip install tabulate


In [ ]:
import sqlite3
from tabulate import tabulate

def students_with_courses():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT Students.student_id, Students.first_name || ' ' || Students.last_name AS student_name,
                   Courses.course_name, Enrollments.grade
            FROM Students
            LEFT JOIN Enrollments ON Students.student_id = Enrollments.student_id
            LEFT JOIN Courses ON Enrollments.course_id = Courses.course_id
            ORDER BY Students.student_id
        """)
        rows = cursor.fetchall()
        headers = ["Student ID", "Student Name", "Course Name", "Grade"]
        print(tabulate(rows, headers, tablefmt="grid"))

def teachers_with_courses():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT Teachers.teacher_id, Teachers.first_name || ' ' || Teachers.last_name AS teacher_name,
                   Courses.course_name
            FROM Teachers
            LEFT JOIN Courses ON Teachers.department_id = Courses.department_id
            ORDER BY Teachers.teacher_id
        """)
        rows = cursor.fetchall()
        headers = ["Teacher ID", "Teacher Name", "Course Name"]
        print(tabulate(rows, headers, tablefmt="grid"))

def count_students_per_course():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT Courses.course_name, COUNT(Enrollments.student_id) AS student_count
            FROM Courses
            LEFT JOIN Enrollments ON Courses.course_id = Enrollments.course_id
            GROUP BY Courses.course_id
        """)
        rows = cursor.fetchall()
        headers = ["Course Name", "Student Count"]
        print(tabulate(rows, headers, tablefmt="grid"))

def students_with_no_enrollments():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT student_id, first_name || ' ' || last_name AS student_name
            FROM Students
            WHERE student_id NOT IN (SELECT DISTINCT student_id FROM Enrollments)
        """)
        rows = cursor.fetchall()
        headers = ["Student ID", "Student Name"]
        print(tabulate(rows, headers, tablefmt="grid"))


In [ ]:
students_with_courses()
teachers_with_courses()
count_students_per_course()
students_with_no_enrollments()


+--------------+----------------+-----------------+---------+
|   Student ID | Student Name   | Course Name     | Grade   |
+==============+================+=================+=========+
|            1 | Alice Smith    | Data Structures | B+      |
+--------------+----------------+-----------------+---------+
|            2 | Bob Johnson    |                 |         |
+--------------+----------------+-----------------+---------+
|            4 | Ethan Wong     |                 |         |
+--------------+----------------+-----------------+---------+
+--------------+----------------+----------------------------------+
|   Teacher ID | Teacher Name   | Course Name                      |
+==============+================+==================================+
|            1 | John Doe       | Algorithms                       |
+--------------+----------------+----------------------------------+
|            1 | John Doe       | Data Structures                  |
+--------------+------------

In [ ]:
pip install bcrypt


In [ ]:
import bcrypt

def create_user(username, password, role):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        # Hash the password
        hashed = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())
        cursor.execute("""
            INSERT INTO Users (username, password, role)
            VALUES (?, ?, ?)
        """, (username, hashed, role))
        conn.commit()
        print(" User created with hashed password.")
def login(username, password):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT password, role FROM Users WHERE username = ?", (username,))
        result = cursor.fetchone()
        if result:
            stored_hash = result[0]
            role = result[1]
            if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
                print(f" Login successful! Role: {role}")
                return True, role
            else:
                print(" Incorrect password.")
                return False, None
        else:
            print(" Username not found.")
            return False, None


In [ ]:
create_user("teacher1", "teachpass123", "teacher")
success, role = login("teacher1", "teachpass123")  # Should print success and role


 User created with hashed password.
 Login successful! Role: teacher


In [ ]:
permissions = {
    "admin": {"create", "read", "update", "delete"},
    "teacher": {"create", "read", "update"},  # Added 'create' here
    "student": {"read"}
}


def requires_permission(action):
    def decorator(func):
        def wrapper(user_role, *args, **kwargs):
            if action in permissions.get(user_role, set()):
                return func(*args, **kwargs)
            else:
                print(f" Permission denied for role '{user_role}' to perform '{action}'.")
                return None
        return wrapper
    return decorator


In [ ]:
@requires_permission("create")
def create_student(first_name, last_name, dob, email, department_id):
    # existing create_student code here
    pass

@requires_permission("read")
def read_students():
    # existing read_students code here
    pass

@requires_permission("update")
def update_student(student_id, first_name, last_name, dob, email, department_id):
    # existing update_student code here
    pass

@requires_permission("delete")
def delete_student(student_id):
    # existing delete_student code here
    pass


In [ ]:
user_role = "teacher"  # example, from login result

create_student(user_role, "Ethan", "Wong", "2002-05-10", "ethan.wong@example.com", 1)  # denied
read_students(user_role)  # allowed


In [ ]:
@requires_permission("create")
def create_student(first_name, last_name, dob, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Students (first_name, last_name, dob, email, department_id)
            VALUES (?, ?, ?, ?, ?)
        """, (first_name, last_name, dob, email, department_id))
        conn.commit()
        print(" Student created.")


In [ ]:
user_role = "admin"  # from login
create_student(user_role, "Ethan", "Wong", "2002-05-10", "ethan.wong@example.com", 1)


 Student created.


In [ ]:
import sqlite3

# --- Permissions and decorator ---
permissions = {
    "admin": {"create", "read", "update", "delete"},
    "teacher": {"create", "read", "update"},
    "student": {"read"}
}

def requires_permission(action):
    def decorator(func):
        def wrapper(user_role, *args, **kwargs):
            if action in permissions.get(user_role, set()):
                return func(*args, **kwargs)
            else:
                print(f" Permission denied for role '{user_role}' to perform '{action}'.")
                return None
        return wrapper
    return decorator

# --- Departments CRUD ---
@requires_permission("create")
def create_department(department_name):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO Departments (department_name) VALUES (?)", (department_name,))
        conn.commit()
        print(" Department created.")

@requires_permission("read")
def read_departments():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Departments")
        for row in cursor.fetchall():
            print(row)

@requires_permission("update")
def update_department(department_id, department_name):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("UPDATE Departments SET department_name=? WHERE department_id=?", (department_name, department_id))
        conn.commit()
        print(" Department updated.")

@requires_permission("delete")
def delete_department(department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Departments WHERE department_id=?", (department_id,))
        conn.commit()
        print(" Department deleted.")

# --- Students CRUD ---
@requires_permission("create")
def create_student(first_name, last_name, dob, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Students (first_name, last_name, dob, email, department_id)
            VALUES (?, ?, ?, ?, ?)
        """, (first_name, last_name, dob, email, department_id))
        conn.commit()
        print(" Student created.")

@requires_permission("read")
def read_students():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Students")
        for row in cursor.fetchall():
            print(row)

@requires_permission("update")
def update_student(student_id, first_name, last_name, dob, email, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Students SET first_name=?, last_name=?, dob=?, email=?, department_id=?
            WHERE student_id=?
        """, (first_name, last_name, dob, email, department_id, student_id))
        conn.commit()
        print(" Student updated.")

@requires_permission("delete")
def delete_student(student_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Students WHERE student_id=?", (student_id,))
        conn.commit()
        print(" Student deleted.")

# --- Teachers CRUD ---
@requires_permission("create")
def create_teacher(first_name, last_name, email, phone_number, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Teachers (first_name, last_name, email, phone_number, department_id)
            VALUES (?, ?, ?, ?, ?)
        """, (first_name, last_name, email, phone_number, department_id))
        conn.commit()
        print(" Teacher created.")

@requires_permission("read")
def read_teachers():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Teachers")
        for row in cursor.fetchall():
            print(row)

@requires_permission("update")
def update_teacher(teacher_id, first_name, last_name, email, phone_number, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Teachers SET first_name=?, last_name=?, email=?, phone_number=?, department_id=?
            WHERE teacher_id=?
        """, (first_name, last_name, email, phone_number, department_id, teacher_id))
        conn.commit()
        print(" Teacher updated.")

@requires_permission("delete")
def delete_teacher(teacher_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Teachers WHERE teacher_id=?", (teacher_id,))
        conn.commit()
        print(" Teacher deleted.")

# --- Courses CRUD ---
@requires_permission("create")
def create_course(course_name, course_description, credits, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Courses (course_name, course_description, credits, department_id)
            VALUES (?, ?, ?, ?)
        """, (course_name, course_description, credits, department_id))
        conn.commit()
        print(" Course created.")

@requires_permission("read")
def read_courses():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Courses")
        for row in cursor.fetchall():
            print(row)

@requires_permission("update")
def update_course(course_id, course_name, course_description, credits, department_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Courses SET course_name=?, course_description=?, credits=?, department_id=?
            WHERE course_id=?
        """, (course_name, course_description, credits, department_id, course_id))
        conn.commit()
        print(" Course updated.")

@requires_permission("delete")
def delete_course(course_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Courses WHERE course_id=?", (course_id,))
        conn.commit()
        print(" Course deleted.")

# --- Enrollments CRUD ---
@requires_permission("create")
def create_enrollment(student_id, course_id, enrollment_date, grade=None):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Enrollments (student_id, course_id, enrollment_date, grade)
            VALUES (?, ?, ?, ?)
        """, (student_id, course_id, enrollment_date, grade))
        conn.commit()
        print(" Enrollment created.")

@requires_permission("read")
def read_enrollments():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Enrollments")
        for row in cursor.fetchall():
            print(row)

@requires_permission("update")
def update_enrollment(enrollment_id, student_id, course_id, enrollment_date, grade):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Enrollments SET student_id=?, course_id=?, enrollment_date=?, grade=?
            WHERE enrollment_id=?
        """, (student_id, course_id, enrollment_date, grade, enrollment_id))
        conn.commit()
        print(" Enrollment updated.")

@requires_permission("delete")
def delete_enrollment(enrollment_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Enrollments WHERE enrollment_id=?", (enrollment_id,))
        conn.commit()
        print(" Enrollment deleted.")

# --- Users CRUD ---
@requires_permission("create")
def create_user(username, password_hash, role):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO Users (username, password_hash, role)
            VALUES (?, ?, ?)
        """, (username, password_hash, role))
        conn.commit()
        print(" User created.")

@requires_permission("read")
def read_users():
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT user_id, username, role FROM Users")
        for row in cursor.fetchall():
            print(row)

@requires_permission("update")
def update_user(user_id, username, password_hash, role):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("""
            UPDATE Users SET username=?, password_hash=?, role=?
            WHERE user_id=?
        """, (username, password_hash, role, user_id))
        conn.commit()
        print(" User updated.")

@requires_permission("delete")
def delete_user(user_id):
    with sqlite3.connect('sims.db') as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Users WHERE user_id=?", (user_id,))
        conn.commit()
        print(" User deleted.")


In [ ]:
user_role = "teacher"
create_student(user_role, "Ethan", "Wong", "2002-05-10", "ethan.wong@example.com", 1)
read_students(user_role)


 Student created.
(1, 'Alice', 'Smith', '2000-01-01', 'alice.updated@example.com', 1, 2)
(2, 'Bob', 'Johnson', '1999-05-12', 'bob@example.com', 2, None)
(4, 'Ethan', 'Wong', '2002-05-10', 'ethan.wong@example.com', 1, None)
(5, 'Ethan', 'Wong', '2002-05-10', 'ethan.wong@example.com', 1, None)
(6, 'Ethan', 'Wong', '2002-05-10', 'ethan.wong@example.com', 1, None)


In [ ]:
DB_NAME = "sims.db"


In [ ]:
def setup_users_table():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Create table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        role TEXT NOT NULL CHECK(role IN ('admin', 'teacher', 'student'))
    );
    """)

    # Insert test users
    sample_users = [
        ('admin1', 'adminpass', 'admin'),
        ('teacher1', 'teachpass', 'teacher'),
        ('student1', 'studpass', 'student'),
    ]

    for user in sample_users:
        try:
            cursor.execute("INSERT INTO users (username, password, role) VALUES (?, ?, ?);", user)
        except sqlite3.IntegrityError:
            pass  # Skip duplicates

    conn.commit()
    conn.close()
    print(" Sample users created.")

setup_users_table()


 Sample users created.


In [ ]:
import sqlite3

DB_NAME = "sims.db"

def print_all_users():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT username, password, role FROM users")

    users = cursor.fetchall()
    conn.close()
    for u in users:
        print(u)

print_all_users()


('admin', 'newadminpass', 'admin')
('alice_smith', 'alicepass', 'student')
('johndoe', 'doepass', 'teacher')
('newuser', 'securepassword', 'student')
('teacher1', b'$2b$12$edePxnoGuAVcZVvc3MzoqudL6lJxq0AjyuYaB2H/fCoaBC1koKKYy', 'teacher')
('admin1', 'adminpass', 'admin')
('student1', 'studpass', 'student')


In [ ]:
import sqlite3

DB_NAME = "sims.db"

def update_teacher_password():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Update teacher1 password to plain text for now
    cursor.execute("UPDATE users SET password=? WHERE username=?", ("teachpass", "teacher1"))

    conn.commit()
    conn.close()
    print(" teacher1 password updated to plain text.")

update_teacher_password()


 teacher1 password updated to plain text.


In [ ]:
conn = sqlite3.connect("sims.db")
cursor = conn.cursor()

# Link student1 user_id to Bob
cursor.execute("""
UPDATE Students
SET user_id = (SELECT user_id FROM Users WHERE username = 'student1')
WHERE email = 'bob@example.com'
""")

conn.commit()
print(" Linked 'student1' to Bob in Students table.")
conn.close()


 Linked 'student1' to Bob in Students table.


In [ ]:
import sqlite3
import getpass
import logging

logging.basicConfig(
    filename="sims.log",
    filemode='a',  # append mode
    format='%(asctime)s %(levelname)s: %(message)s',
    level=logging.DEBUG
)

logger = logging.getLogger()
logger.info("Logging setup complete and writing to sims.log")


DB_NAME = "sims.db"


def connect_db():
    return sqlite3.connect("sims.db")

def get_user_role(username):
    conn = connect_db()
    cursor = conn.cursor()
    cursor.execute("SELECT role FROM users WHERE username = ?", (username,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return row[0]  # role string like 'admin', 'teacher', or 'student'
    else:
        return None

def check_duplicates():
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("""
            SELECT first_name, last_name, dob, email, COUNT(*)
            FROM students
            GROUP BY first_name, last_name, dob, email
            HAVING COUNT(*) > 1
        """)
        duplicates = cursor.fetchall()
        conn.close()

        if duplicates:
            logger.info(f"Found {len(duplicates)} duplicate student groups.")
            print("\n Duplicate students found:")
            for d in duplicates:
                print(f"{d[0]} {d[1]}, DOB: {d[2]}, Email: {d[3]} — Count: {d[4]}")
        else:
            logger.info("No duplicate students found.")
            print("\nNo duplicate students found.")
    except Exception as e:
        logger.error(f"Error checking duplicates: {e}")
        print("An error occurred while checking duplicates.")

def remove_duplicate_students():
    try:
        conn = connect_db()
        cursor = conn.cursor()

        cursor.execute("""
            SELECT MIN(student_id) as keep_id, first_name, last_name, dob, email
            FROM students
            GROUP BY first_name, last_name, dob, email
            HAVING COUNT(*) > 1
        """)
        duplicates = cursor.fetchall()

        total_removed = 0

        for keep_id, first_name, last_name, dob, email in duplicates:
            cursor.execute("""
                DELETE FROM students
                WHERE first_name=? AND last_name=? AND dob=? AND email=? AND student_id <> ?
            """, (first_name, last_name, dob, email, keep_id))
            total_removed += cursor.rowcount

        conn.commit()
        conn.close()

        logger.info(f"Removed {total_removed} duplicate student records.")
        print(f"\n Removed {total_removed} duplicate student records.")
    except Exception as e:
        logger.error(f"Error removing duplicates: {e}")
        print("An error occurred while removing duplicates.")



def list_students():
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("SELECT student_id, first_name, last_name, dob, email FROM students")
        students = cursor.fetchall()

        if students:
            print("\nID | First Name | Last Name | DOB        | Email")
            print("-" * 50)
            for s in students:
                print(f"{s[0]:<3} | {s[1]:<10} | {s[2]:<9} | {s[3]:<10} | {s[4]}")
            logger.info(f"Listed {len(students)} students.")
        else:
            print("No students found.")
            logger.info("Listed students: no records found.")

        input("\nPress Enter to return to Manage Students menu...")

    except Exception as e:
        print("An error occurred while listing students.")
        logger.error(f"Error in list_students: {e}")
    finally:
        if 'conn' in locals():
            conn.close()


def add_student():
    print("\n--- Add New Student ---")
    try:
        first_name = input("First name: ").strip()
        last_name = input("Last name: ").strip()
        dob = input("Date of Birth (YYYY-MM-DD): ").strip()
        email = input("Email: ").strip()

        if not first_name or not last_name or not dob or not email:
            print("All fields are required.")
            logger.warning("Attempted to add student with missing fields.")
            return

        conn = connect_db()
        cursor = conn.cursor()

        # Check for duplicates
        cursor.execute("""
            SELECT * FROM students WHERE first_name=? AND last_name=? AND dob=? AND email=?
        """, (first_name, last_name, dob, email))
        exists = cursor.fetchone()

        if exists:
            print("Student already exists. Aborting insert.")
            logger.info(f"Duplicate student not added: {first_name} {last_name}, DOB: {dob}, Email: {email}")
        else:
            cursor.execute("""
                INSERT INTO students (first_name, last_name, dob, email) VALUES (?, ?, ?, ?)
            """, (first_name, last_name, dob, email))
            conn.commit()
            print("Student added successfully.")
            logger.info(f"Added student: {first_name} {last_name}, DOB: {dob}, Email: {email}")
    except Exception as e:
        print("An error occurred while adding the student.")
        logger.error(f"Error in add_student: {e}")
    finally:
        if 'conn' in locals():
            conn.close()

def update_student():
    print("\n--- Update Student ---")
    try:
        student_id = input("Enter Student ID to update: ").strip()
        if not student_id.isdigit():
            print("Invalid ID. Must be a number.")
            logger.warning(f"Invalid student ID entered for update: {student_id}")
            return

        conn = connect_db()
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
        student = cursor.fetchone()
        if not student:
            print("Student not found.")
            logger.info(f"Update attempted on non-existent student ID: {student_id}")
            return

        print(f"Current info: First Name: {student[1]}, Last Name: {student[2]}, DOB: {student[3]}, Email: {student[4]}")

        new_first = input("New First Name (press enter to keep current): ").strip()
        new_last = input("New Last Name (press enter to keep current): ").strip()
        new_dob = input("New DOB (YYYY-MM-DD) (press enter to keep current): ").strip()
        new_email = input("New Email (press enter to keep current): ").strip()

        first_name = new_first if new_first else student[1]
        last_name = new_last if new_last else student[2]
        dob = new_dob if new_dob else student[3]
        email = new_email if new_email else student[4]

        cursor.execute("""
            UPDATE students
            SET first_name = ?, last_name = ?, dob = ?, email = ?
            WHERE student_id = ?
        """, (first_name, last_name, dob, email, student_id))
        conn.commit()
        print("Student updated successfully.")
        logger.info(f"Student ID {student_id} updated to: {first_name} {last_name}, DOB: {dob}, Email: {email}")

    except Exception as e:
        print("An error occurred while updating the student.")
        logger.error(f"Error in update_student (ID: {student_id}): {e}")
    finally:
        if 'conn' in locals():
            conn.close()


def delete_student():
    print("\n--- Delete Student ---")
    try:
        student_id = input("Enter Student ID to delete: ").strip()
        if not student_id.isdigit():
            print("Invalid ID. Must be a number.")
            logger.warning(f"Invalid student ID entered for delete: {student_id}")
            return

        conn = connect_db()
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
        student = cursor.fetchone()
        if not student:
            print("Student not found.")
            logger.info(f"Delete attempted on non-existent student ID: {student_id}")
            return

        confirm = input(f"Are you sure you want to delete student {student[1]} {student[2]}? (y/n): ").strip().lower()
        if confirm == 'y':
            cursor.execute("DELETE FROM students WHERE student_id = ?", (student_id,))
            conn.commit()
            print("Student deleted successfully.")
            logger.info(f"Deleted student ID {student_id}: {student[1]} {student[2]}")
        else:
            print("Delete operation cancelled.")
            logger.info(f"Delete cancelled for student ID {student_id}")

    except Exception as e:
        print("An error occurred while deleting the student.")
        logger.error(f"Error in delete_student (ID: {student_id}): {e}")
    finally:
        if 'conn' in locals():
            conn.close()


def manage_students():
    while True:
        print("\n--- Manage Students ---")
        print("1. List all students")
        print("2. Add new student")
        print("3. Update student")
        print("4. Delete student")
        print("5. Back to Admin Menu")

        choice = input("Choose option: ").strip()

        if choice == '1':
            list_students()
        elif choice == '2':
            add_student()
        elif choice == '3':
            update_student()
        elif choice == '4':
            delete_student()
        elif choice == '5':
            break
        else:
            print("Feature coming soon or invalid choice. Please try again.")

def check_students_count():
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM students")
        count = cursor.fetchone()[0]
        conn.close()
        print(f"Total students in DB: {count}")
    except Exception as e:
        logger.error(f"Error checking student count: {e}")
        print("An error occurred while fetching student count.")


def student_menu(student_id):
    logger.info(f"Student {student_id} accessed the student menu.")
    while True:
        print("\n--- Student Menu ---")
        print("1. View my profile")
        print("2. View my courses")
        print("3. Logout")

        choice = input("Choose option: ").strip()

        if choice == '1':
            logger.info(f"Student {student_id} selected 'View my profile'.")
            view_student_profile(student_id)
        elif choice == '2':
            logger.info(f"Student {student_id} selected 'View my courses'.")
            view_student_courses(student_id)
        elif choice == '3':
            logger.info(f"Student {student_id} logged out.")
            print("Logging out...")
            break
        else:
            logger.warning(f"Student {student_id} entered invalid option: {choice}")
            print("Invalid choice. Please try again.")


def view_student_profile(student_id):
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("SELECT student_id, first_name, last_name, dob, email FROM students WHERE student_id=?", (student_id,))
        student = cursor.fetchone()
        conn.close()

        if student:
            print(f"\nStudent ID: {student[0]}")
            print(f"Name: {student[1]} {student[2]}")
            print(f"Date of Birth: {student[3]}")
            print(f"Email: {student[4]}")
            logger.info(f"Viewed profile for student ID {student_id}")
        else:
            print("Student not found.")
            logger.info(f"Student profile not found for ID {student_id}")

    except Exception as e:
        print("An error occurred while retrieving the student profile.")
        logger.error(f"Error viewing profile for student ID {student_id}: {e}")


def view_student_courses(student_id):
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("""
            SELECT c.course_id, c.course_name, c.department_id
            FROM courses c
            JOIN enrollments e ON c.course_id = e.course_id
            WHERE e.student_id = ?
        """, (student_id,))
        courses = cursor.fetchall()
        conn.close()

        if courses:
            print("\nMy Courses:")
            print("ID | Course Name | Department ID")
            print("------------------------------")
            for c in courses:
                print(f"{c[0]} | {c[1]} | {c[2]}")
            logger.info(f"Student ID {student_id} viewed {len(courses)} courses.")
        else:
            print("You are not enrolled in any courses.")
            logger.info(f"Student ID {student_id} has no enrolled courses.")

    except Exception as e:
        print("An error occurred while retrieving courses.")
        logger.error(f"Error in view_student_courses for student {student_id}: {e}")


def get_student_id(username):
    try:
        conn = sqlite3.connect("sims.db")
        cursor = conn.cursor()
        cursor.execute("""
            SELECT s.student_id
            FROM Students s
            JOIN Users u ON s.user_id = u.user_id
            WHERE u.username = ?
        """, (username,))
        result = cursor.fetchone()
        conn.close()

        if result:
            logger.info(f"Username '{username}' matched to student ID {result[0]}")
        else:
            logger.warning(f"Username '{username}' not found in get_student_id.")

        return result[0] if result else None

    except Exception as e:
        logger.error(f"Error in get_student_id for username '{username}': {e}")
        return None


def manage_teachers():
    while True:
        print("\n--- Manage Teachers ---")
        print("1. List all teachers")
        print("2. Add new teacher")
        print("3. Update teacher")
        print("4. Delete teacher")
        print("5. Back to Admin Menu")

        choice = input("Choose option: ").strip()

        if choice == '1':
            list_teachers()
        elif choice == '2':
            add_teacher()
        elif choice == '3':
            update_teacher()
        elif choice == '4':
            delete_teacher()
        elif choice == '5':
            break
        else:
            print("Invalid choice. Please try again.")


def list_teachers():
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("SELECT teacher_id, first_name, last_name, email, department_id FROM teachers")
        teachers = cursor.fetchall()
        conn.close()

        if not teachers:
            print("No teachers found.")
            logger.info("Listed teachers: no records found.")
            return

        print("\nID | First Name | Last Name | Email                   | Department ID")
        print("-" * 70)
        for t in teachers:
            print(f"{t[0]:<3} | {t[1]:<10} | {t[2]:<9} | {t[3]:<25} | {t[4]}")
        logger.info(f"Listed {len(teachers)} teachers.")
        input("\nPress Enter to return to Manage Teachers menu...")

    except Exception as e:
        print("An error occurred while listing teachers.")
        logger.error(f"Error in list_teachers: {e}")

def add_teacher():
    print("\n--- Add New Teacher ---")
    first_name = input("First name: ").strip()
    last_name = input("Last name: ").strip()
    email = input("Email: ").strip()
    department_id = input("Department ID: ").strip()

    try:
        conn = connect_db()
        cursor = conn.cursor()

        cursor.execute("""
            SELECT * FROM teachers WHERE first_name=? AND last_name=? AND email=?
        """, (first_name, last_name, email))

        if cursor.fetchone():
            print("Teacher with the same name and email already exists.")
            logger.warning(f"Attempted to add duplicate teacher: {first_name} {last_name}, {email}")
            conn.close()
            return

        cursor.execute("""
            INSERT INTO teachers (first_name, last_name, email, department_id)
            VALUES (?, ?, ?, ?)
        """, (first_name, last_name, email, department_id))

        conn.commit()
        conn.close()
        print("Teacher added successfully.")
        logger.info(f"Added teacher: {first_name} {last_name}, {email}, Dept: {department_id}")

    except Exception as e:
        print("An error occurred while adding a teacher.")
        logger.error(f"Error in add_teacher: {e}")



def update_teacher():
    print("\n--- Update Teacher ---")
    teacher_id = input("Enter Teacher ID to update: ").strip()
    if not teacher_id.isdigit():
        print("Invalid ID. Must be a number.")
        logger.warning(f"Invalid teacher ID input for update: {teacher_id}")
        return

    try:
        conn = connect_db()
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM teachers WHERE teacher_id = ?", (teacher_id,))
        teacher = cursor.fetchone()
        if not teacher:
            print("Teacher not found.")
            logger.info(f"Update attempted for non-existing teacher ID: {teacher_id}")
            conn.close()
            return

        print(f"Current info: First Name: {teacher[1]}, Last Name: {teacher[2]}, DOB: {teacher[3]}, Email: {teacher[4]}")

        new_first = input("New First Name (press enter to keep current): ").strip()
        new_last = input("New Last Name (press enter to keep current): ").strip()
        new_dob = input("New DOB (YYYY-MM-DD) (press enter to keep current): ").strip()
        new_email = input("New Email (press enter to keep current): ").strip()

        first_name = new_first if new_first else teacher[1]
        last_name = new_last if new_last else teacher[2]
        dob = new_dob if new_dob else teacher[3]
        email = new_email if new_email else teacher[4]

        cursor.execute("""
            UPDATE teachers
            SET first_name = ?, last_name = ?, dob = ?, email = ?
            WHERE teacher_id = ?
        """, (first_name, last_name, dob, email, teacher_id))
        conn.commit()
        conn.close()
        print("Teacher updated successfully.")
        logger.info(f"Updated teacher ID {teacher_id} to {first_name} {last_name}, DOB: {dob}, Email: {email}")

    except Exception as e:
        print("An error occurred while updating teacher.")
        logger.error(f"Error in update_teacher (ID {teacher_id}): {e}")

def delete_teacher():
    print("\n--- Delete Teacher ---")
    teacher_id = input("Enter Teacher ID to delete: ").strip()
    if not teacher_id.isdigit():
        print("Invalid ID. Must be a number.")
        logger.warning(f"Invalid teacher ID input for delete: {teacher_id}")
        return

    try:
        conn = connect_db()
        cursor = conn.cursor()

        cursor.execute("SELECT * FROM teachers WHERE teacher_id = ?", (teacher_id,))
        teacher = cursor.fetchone()
        if not teacher:
            print("Teacher not found.")
            logger.info(f"Delete attempted for non-existing teacher ID: {teacher_id}")
            conn.close()
            return

        confirm = input(f"Are you sure you want to delete teacher {teacher[1]} {teacher[2]}? (y/n): ").strip().lower()
        if confirm == 'y':
            cursor.execute("DELETE FROM teachers WHERE teacher_id = ?", (teacher_id,))
            conn.commit()
            print("Teacher deleted successfully.")
            logger.info(f"Deleted teacher ID {teacher_id} - {teacher[1]} {teacher[2]}")
        else:
            print("Delete operation cancelled.")
            logger.info(f"Delete operation cancelled for teacher ID {teacher_id}")

        conn.close()

    except Exception as e:
        print("An error occurred while deleting teacher.")
        logger.error(f"Error in delete_teacher (ID {teacher_id}): {e}")

def check_duplicates_teachers():
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("""
            SELECT first_name, last_name, email, COUNT(*) as count
            FROM teachers
            GROUP BY first_name, last_name, email
            HAVING count > 1
        """)

        duplicates = cursor.fetchall()
        conn.close()

        if duplicates:
            print("\n Duplicate teachers found:")
            for d in duplicates:
                print(f"{d[0]} {d[1]}, Email: {d[2]} — Count: {d[3]}")
            logger.warning(f"Duplicate teachers detected: {len(duplicates)} groups")
        else:
            print("\nNo duplicate teachers found.")
            logger.info("Checked duplicates in teachers: none found")

    except Exception as e:
        print("An error occurred while checking duplicate teachers.")
        logger.error(f"Error in check_duplicates_teachers: {e}")

def remove_duplicate_teachers():
    try:
        conn = connect_db()
        cursor = conn.cursor()
        cursor.execute("""
            DELETE FROM teachers
            WHERE teacher_id NOT IN (
                SELECT MIN(teacher_id)
                FROM teachers
                GROUP BY first_name, last_name, email
            )
        """)
        removed = cursor.rowcount
        conn.commit()
        conn.close()
        print(f"Removed {removed} duplicate teacher records.")
        logger.info(f"Removed {removed} duplicate teacher records.")

    except Exception as e:
        print("An error occurred while removing duplicate teachers.")
        logger.error(f"Error in remove_duplicate_teachers: {e}")

def list_courses():
    conn = connect_db()
    cursor = conn.cursor()
    cursor.execute("SELECT course_id, course_name, department_id FROM courses")
    courses = cursor.fetchall()
    conn.close()

    print("\nID | Course Name           | Department ID")
    print("-------------------------------------------")
    for c in courses:
        print(f"{c[0]:<3} | {c[1]:<20} | {c[2]}")

    logger.info(f"Listed {len(courses)} courses")  # optional

    input("\nPress Enter to return to Teacher Menu...")


def teacher_menu():
    logger.info("Teacher entered the Teacher Menu.")
    while True:
        print("\n--- Teacher Menu ---")
        print("1. View students")
        print("2. View courses")
        print("3. Logout")

        choice = input("Choose option: ").strip()

        if choice == '1':
            logger.info("Teacher selected: View students")
            list_students()  # or your actual function
        elif choice == '2':
            logger.info("Teacher selected: View courses")
            list_courses()
        elif choice == '3':
            logger.info("Teacher selected: Logout")
            print("Logging out...")
            break
        else:
            logger.warning(f"Teacher entered invalid option: {choice}")
            print("Invalid choice. Try again.")


def manage_courses(conn):
    while True:
        print("\n====== COURSE MANAGEMENT ======")
        print("1. View All Courses")
        print("2. Add a New Course")
        print("3. Update an Existing Course")
        print("4. Delete a Course")
        print("0. Back to Main Menu")

        choice = input("Select an option: ")

        if choice == '1':
            courses = get_all_courses(conn)
            print("\n--- List of Courses ---")
            for c in courses:
                print(f"ID: {c[0]}, Name: {c[1]}, Department ID: {c[2]}")
        elif choice == '2':
            add_course(conn)
        elif choice == '3':
            update_course(conn)
        elif choice == '4':
            delete_course(conn)
        elif choice == '0':
            break
        else:
            print(" Invalid choice. Try again.")


def get_all_courses(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT course_id, course_name, department_id FROM courses")
    courses = cursor.fetchall()
    logger.info(f"Fetched {len(courses)} courses from DB")
    return courses

def add_course(conn):
    try:
        course_name = input("Enter course name: ").strip()
        department_id = input("Enter department ID: ").strip()
        cursor = conn.cursor()
        cursor.execute("INSERT INTO courses (course_name, department_id) VALUES (?, ?)", (course_name, department_id))
        conn.commit()
        print(" Course added successfully.")
        logger.info(f"Added course: {course_name} under department {department_id}")
    except Exception as e:
        logger.error(f"Error adding course: {e}")
        print(" Failed to add course.")

def update_course(conn):
    try:
        course_id = input("Enter the Course ID to update: ").strip()
        new_name = input("Enter new course name: ").strip()
        new_department_id = input("Enter new department ID: ").strip()
        cursor = conn.cursor()
        cursor.execute("UPDATE courses SET course_name = ?, department_id = ? WHERE course_id = ?", (new_name, new_department_id, course_id))
        if cursor.rowcount == 0:
            print(" No course found with that ID.")
            logger.warning(f"Update attempt failed; no course with ID {course_id}")
        else:
            conn.commit()
            print(" Course updated successfully.")
            logger.info(f"Updated course ID {course_id} to name '{new_name}', department {new_department_id}")
    except Exception as e:
        logger.error(f"Error updating course: {e}")
        print(" Failed to update course.")

def delete_course(conn):
    try:
        course_id = input("Enter the Course ID to delete: ").strip()
        cursor = conn.cursor()
        cursor.execute("DELETE FROM courses WHERE course_id = ?", (course_id,))
        if cursor.rowcount == 0:
            print(" No course found with that ID.")
            logger.warning(f"Delete attempt failed; no course with ID {course_id}")
        else:
            conn.commit()
            print(" Course deleted successfully.")
            logger.info(f"Deleted course ID {course_id}")
    except Exception as e:
        logger.error(f"Error deleting course: {e}")
        print(" Failed to delete course.")


def get_enrollments_by_student(conn, student_id):
    cursor = conn.cursor()
    query = """
    SELECT e.enrollment_id, c.course_id, c.course_name, d.department_name
    FROM enrollments e
    JOIN courses c ON e.course_id = c.course_id
    JOIN departments d ON c.department_id = d.department_id
    WHERE e.student_id = ?
    """
    cursor.execute(query, (student_id,))
    enrollments = cursor.fetchall()
    logger.debug(f"Fetched {len(enrollments)} enrollments for student_id={student_id}")
    return enrollments

def add_enrollment(conn, student_id, course_id):
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM enrollments WHERE student_id = ? AND course_id = ?", (student_id, course_id))
    if cursor.fetchone():
        logger.warning(f"Duplicate enrollment attempt for student_id={student_id} course_id={course_id}")
        return False, "Student already enrolled in this course."

    cursor.execute("INSERT INTO enrollments (student_id, course_id) VALUES (?, ?)", (student_id, course_id))
    conn.commit()
    logger.info(f"Enrollment added for student_id={student_id} course_id={course_id}")
    return True, "Enrollment added successfully."

def delete_enrollment(conn, enrollment_id):
    cursor = conn.cursor()
    cursor.execute("DELETE FROM enrollments WHERE enrollment_id = ?", (enrollment_id,))
    deleted = cursor.rowcount > 0
    if deleted:
        logger.info(f"Enrollment deleted with enrollment_id={enrollment_id}")
    else:
        logger.warning(f"Attempted to delete non-existent enrollment_id={enrollment_id}")
    conn.commit()
    return deleted


def manage_enrollments(conn):
    while True:
        print("\n====== ENROLLMENT MANAGEMENT ======")
        print("1. View Student's Enrolled Courses")
        print("2. Add Enrollment")
        print("3. Delete Enrollment")
        print("0. Back to Main Menu")
        choice = input("Select an option: ")

        if choice == '1':
            student_id = int(input("Enter student ID: "))
            enrollments = get_enrollments_by_student(conn, student_id)
            if enrollments:
                print(f"\nCourses enrolled by Student ID {student_id}:")
                for e in enrollments:
                    print(f"Enrollment ID: {e[0]}, Course ID: {e[1]}, Name: {e[2]}, Department: {e[3]}")
            else:
                print("No courses enrolled for this student.")

        elif choice == '2':
            student_id = int(input("Enter student ID: "))
            course_id = int(input("Enter course ID to enroll: "))
            success, msg = add_enrollment(conn, student_id, course_id)
            print(msg)

        elif choice == '3':
            enrollment_id = int(input("Enter Enrollment ID to delete: "))
            if delete_enrollment(conn, enrollment_id):
                print("Enrollment deleted successfully.")
            else:
                print("Enrollment ID not found.")

        elif choice == '0':
            break

        else:
            print("Invalid option. Please try again.")

def get_all_users(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT user_id, username, role FROM users")
    users = cursor.fetchall()
    logger.debug(f"Retrieved {len(users)} users from database")
    return users

def add_user(conn, username, password, role):
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO users (username, password, role) VALUES (?, ?, ?)", (username, password, role))
        conn.commit()
        logger.info(f"Added new user: username={username}, role={role}")
    except Exception as e:
        logger.error(f"Error adding user {username}: {e}")
        raise

def update_user(conn, user_id, username=None, password=None, role=None):
    cursor = conn.cursor()
    updates = []
    params = []

    if username is not None:
        updates.append("username = ?")
        params.append(username)
    if password is not None:
        updates.append("password = ?")
        params.append(password)
    if role is not None:
        updates.append("role = ?")
        params.append(role)

    if not updates:
        logger.warning(f"No update fields provided for user_id={user_id}")
        return

    params.append(user_id)
    query = "UPDATE users SET " + ", ".join(updates) + " WHERE user_id = ?"

    try:
        cursor.execute(query, params)
        conn.commit()
        logger.info(f"Updated user_id={user_id} with fields {updates}")
    except Exception as e:
        logger.error(f"Error updating user_id={user_id}: {e}")
        raise

def delete_user(conn, user_id):
    cursor = conn.cursor()
    cursor.execute("DELETE FROM users WHERE user_id = ?", (user_id,))
    deleted = cursor.rowcount > 0
    conn.commit()
    if deleted:
        logger.info(f"Deleted user with user_id={user_id}")
    else:
        logger.warning(f"Tried to delete non-existent user_id={user_id}")


def manage_users(conn):
    while True:
        print("\n====== USER MANAGEMENT ======")
        print("1. View All Users")
        print("2. Add New User")
        print("3. Update Existing User")
        print("4. Delete User")
        print("0. Back to Main Menu")
        choice = input("Select an option: ")

        if choice == '1':
            users = get_all_users(conn)
            print("\n--- List of Users ---")
            for user in users:
                print(f"ID: {user[0]}, Username: {user[1]}, Role: {user[2]}")
        elif choice == '2':
            username = input("Enter username: ")
            password = input("Enter password: ")
            role = input("Enter role (ADMIN, STUDENT, TEACHER): ").upper()
            if role not in ('ADMIN', 'STUDENT', 'TEACHER'):
                print("Invalid role. Try again.")
            else:
                add_user(conn, username, password, role)
                print("User added successfully.")
        elif choice == '3':
            user_id = int(input("Enter user ID to update: "))
            username = input("Enter new username (leave blank to skip): ").strip() or None
            password = input("Enter new password (leave blank to skip): ").strip() or None
            role = input("Enter new role (ADMIN, STUDENT, TEACHER) (leave blank to skip): ").upper() or None
            if role and role not in ('ADMIN', 'STUDENT', 'TEACHER'):
                print("Invalid role. Try again.")
            else:
                update_user(conn, user_id, username, password, role)
                print("User updated successfully.")
        elif choice == '4':
            user_id = int(input("Enter user ID to delete: "))
            delete_user(conn, user_id)
            print("User deleted successfully.")
        elif choice == '0':
            break
        else:
            print("Invalid option. Please try again.")

def get_all_departments(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT department_id, department_name FROM departments")
    departments = cursor.fetchall()
    logger.debug(f"Fetched {len(departments)} departments")
    return departments

def add_department(conn, department_name):
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO departments (department_name) VALUES (?)", (department_name,))
        conn.commit()
        logger.info(f"Added department: {department_name}")
    except Exception as e:
        logger.error(f"Failed to add department '{department_name}': {e}")
        raise

def update_department(conn, department_id, new_name):
    cursor = conn.cursor()
    try:
        cursor.execute("UPDATE departments SET department_name = ? WHERE department_id = ?", (new_name, department_id))
        if cursor.rowcount == 0:
            logger.warning(f"No department found with ID {department_id} to update")
        else:
            logger.info(f"Updated department ID {department_id} with new name '{new_name}'")
        conn.commit()
    except Exception as e:
        logger.error(f"Failed to update department ID {department_id}: {e}")
        raise

def delete_department(conn, department_id):
    cursor = conn.cursor()
    try:
        cursor.execute("DELETE FROM departments WHERE department_id = ?", (department_id,))
        if cursor.rowcount == 0:
            logger.warning(f"No department found with ID {department_id} to delete")
        else:
            logger.info(f"Deleted department ID {department_id}")
        conn.commit()
    except Exception as e:
        logger.error(f"Failed to delete department ID {department_id}: {e}")
        raise

def manage_departments(conn):
    while True:
        print("\n====== DEPARTMENT MANAGEMENT ======")
        print("1. View All Departments")
        print("2. Add New Department")
        print("3. Update Department")
        print("4. Delete Department")
        print("0. Back to Main Menu")
        choice = input("Select an option: ")

        if choice == '1':
            departments = get_all_departments(conn)
            print("\n--- List of Departments ---")
            for d in departments:
                print(f"ID: {d[0]}, Name: {d[1]}")
        elif choice == '2':
            name = input("Enter new department name: ")
            add_department(conn, name)
            print("Department added successfully.")
        elif choice == '3':
            dep_id = int(input("Enter department ID to update: "))
            new_name = input("Enter new department name: ")
            update_department(conn, dep_id, new_name)
            print("Department updated successfully.")
        elif choice == '4':
            dep_id = int(input("Enter department ID to delete: "))
            delete_department(conn, dep_id)
            print("Department deleted successfully.")
        elif choice == '0':
            break
        else:
            print("Invalid option. Please try again.")

def show_basic_stats():
    conn = connect_db()
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM students")
    student_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM courses")
    course_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM enrollments")
    enrollment_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM departments")
    dept_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM teachers")
    teacher_count = cursor.fetchone()[0]

    conn.close()

    print("\n====== SYSTEM STATISTICS ======")
    print(f" Total Students: {student_count}")
    print(f" Total Courses: {course_count}")
    print(f" Total Enrollments: {enrollment_count}")
    print(f" Total Departments: {dept_count}")
    print(f" Total Teachers: {teacher_count}")



def admin_menu():
    logger.info("Admin entered the Admin Menu.")
    conn = connect_db()
    try:
        while True:
            print("\n--- Admin Menu ---")
            print("1. Manage Students")
            print("2. Manage Teachers")
            print("3. Manage Courses")
            print("4. Manage Users")
            print("5. Manage Enrollments")
            print("6. Manage Departments")
            print("7. View System Statistics")
            print("8. Logout")

            choice = input("Enter option number: ").strip()

            if choice == '1':
                logger.info("Admin selected: Manage Students")
                manage_students()
            elif choice == '2':
                logger.info("Admin selected: Manage Teachers")
                manage_teachers()
            elif choice == '3':
                logger.info("Admin selected: Manage Courses")
                # Open and close connection inside manage_courses
                conn_course = connect_db()
                manage_courses(conn_course)
                conn_course.close()
            elif choice == '4':
                logger.info("Admin selected: Manage Users")
                manage_users(conn)
            elif choice == '5':
                logger.info("Admin selected: Manage Enrollments")
                manage_enrollments(conn)
            elif choice == '6':
                logger.info("Admin selected: Manage Departments")
                manage_departments(conn)
            elif choice == '7':
                logger.info("Admin selected: View System Statistics")
                show_basic_stats()
            elif choice == '8':
                logger.info("Admin selected: Logout")
                print("Logging out...")
                break
            else:
                logger.warning(f"Admin entered invalid option: {choice}")
                print("Feature coming soon or invalid choice. Please try again.")
    finally:
        conn.close()


def main_menu(role):
    logger.info(f"User logged in with role: {role}")
    print(f"\nWelcome! You are logged in as: {role.upper()}")

    if role == "admin":
        logger.info("Entering admin menu")
        admin_menu()

    elif role == "teacher":
        logger.info("Checking duplicate teachers")
        check_duplicates_teachers()
        confirm = input("\nDo you want to remove duplicate teachers now? (y/n): ").strip().lower()
        logger.info(f"Teacher duplicate removal confirmation: {confirm}")
        if confirm == 'y':
            remove_duplicate_teachers()
            logger.info("Duplicate teachers removed")
        else:
            print("Skipping duplicate removal.")
            logger.info("Skipped duplicate teacher removal")
        teacher_menu()

    elif role == "student":
        logger.info("Checking duplicate students")
        check_duplicates()
        confirm = input("\nDo you want to remove duplicate students now? (y/n): ").strip().lower()
        logger.info(f"Student duplicate removal confirmation: {confirm}")
        if confirm == 'y':
            remove_duplicate_students()
            logger.info("Duplicate students removed")
        else:
            print("Skipping duplicate removal.")
            logger.info("Skipped duplicate student removal")
        student_menu()

    else:
        logger.warning(f"Invalid role passed to main_menu: {role}")
        print("Invalid role. Exiting...")

def verify_password(username, input_password):
    conn = connect_db()
    cursor = conn.cursor()
    cursor.execute("SELECT password FROM Users WHERE username = ?", (username,))
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return False
    stored_password = row[0]
    # For now, if you store plain text passwords:
    return stored_password == input_password
    # Or use hashed passwords and verify accordingly



def login_screen():
    username = input("Username: ")
    password = input("Password: ")
    logger.info(f"Login attempt for username: {username}")

    user_role = get_user_role(username)
    if not user_role:
        logger.warning(f"Login failed for username: {username} - user does not exist")
        print("Invalid username or user does not exist.")
        return

    if not verify_password(username, password):
        logger.warning(f"Login failed for username: {username} - invalid password")
        print("Invalid password.")
        return

    logger.info(f"Login successful for username: {username} with role: {user_role}")
    print(f"\nWelcome! You are logged in as: {user_role.upper()}")

    if user_role == "admin":
        admin_menu()
    elif user_role == "teacher":
        teacher_menu()
    elif user_role == "student":
        student_id = get_student_id(username)
        if student_id is not None:
            student_menu(student_id)
        else:
            logger.error(f"Student ID not found for username: {username}")
            print("Student ID not found for this username.")
    else:
        logger.error(f"Unknown user role: {user_role} for username: {username}")
        print("Unknown user role. Exiting.")

if __name__ == "__main__":
    login_screen()



Username: admin1
Password: adminpass

Welcome! You are logged in as: ADMIN

--- Admin Menu ---
1. Manage Students
2. Manage Teachers
3. Manage Courses
4. Manage Users
5. Manage Enrollments
6. Manage Departments
7. View System Statistics
8. Logout
Enter option number: 7

====== SYSTEM STATISTICS ======
 Total Students: 5
 Total Courses: 4
 Total Enrollments: 1
 Total Departments: 4
 Total Teachers: 1

--- Admin Menu ---
1. Manage Students
2. Manage Teachers
3. Manage Courses
4. Manage Users
5. Manage Enrollments
6. Manage Departments
7. View System Statistics
8. Logout
Enter option number: 8
Logging out...


In [ ]:
import sqlite3
import pandas as pd

def export_table_to_csv(table_name, db_path="sims.db"):
    try:
        conn = sqlite3.connect(db_path)
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        conn.close()
        file_name = f"{table_name}.csv"
        df.to_csv(file_name, index=False)
        print(f" Exported {table_name} to {file_name}")
    except Exception as e:
        print(f" Error exporting {table_name}: {e}")


In [ ]:
tables = ['students', 'teachers', 'courses', 'departments', 'enrollments', 'users']
for table in tables:
    export_table_to_csv(table)


 Exported students to students.csv
 Exported teachers to teachers.csv
 Exported courses to courses.csv
 Exported departments to departments.csv
 Exported enrollments to enrollments.csv
 Exported users to users.csv


# Student Information Management System (SIMS)

A CLI-based system built with Python and SQLite for managing student, teacher, course, department, enrollment, and user data. Suitable for academic or system analytics projects.

## 🔧 Features

- **Role-based login system** (Admin, Teacher, Student)
- **Full CRUD operations** for:
  - Students
  - Teachers
  - Courses
  - Departments
  - Users
  - Enrollments
- **Duplicate detection and removal**
- **Logging system** using `logging` module
- **Reports** and future CSV export capabilities
- Simple, modular CLI without web or GUI overhead

## 👤 Roles

- **Admin**:
  - Manage all entities
  - View reports, remove duplicates
- **Teacher**:
  - View courses and students
  - Remove duplicate teacher records
- **Student**:
  - View profile and enrolled courses
  - Remove duplicate student records

## 📁 Database Schema

- `Users(user_id, username, password, role)`
- `Students(student_id, first_name, last_name, dob, email, user_id)`
- `Teachers(teacher_id, first_name, last_name, email, user_id)`
- `Departments(department_id, department_name)`
- `Courses(course_id, course_name, department_id)`
- `Enrollments(enrollment_id, student_id, course_id, enrollment_date, grade)`

## ▶️ Getting Started

1. Ensure Python 3 is installed
2. Run the script:
   ```bash
   python sims_main.py


Login in credentials: -

username: admin1

password: adminpass

or

username: student1

password: studpass

or

username: teacher1

password: teachpass



CSV Export

Use this utility function in your code to export any table:

export_table_to_csv("students")

This creates a students.csv file with all current records.


Conclusion:


---

### Changes Made:
- Completed the “Getting Started” section (it was cut off).
- Added spacing, clarity, and polish in language.
- Clarified that CSV export and logging are **ready to use**.
- Finalized with a conclusion for context.

You're welcome to use this version exactly as-is, or tweak it further. Let me know if you want it saved as a `.md` file for GitHub or submission.
